# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load, explore, and conduct basic analyses on the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj), following best practices for referencing Croissant entities using their `@id` fields. We use the [mlcroissant](https://pypi.org/project/mlcroissant/) library for all data access and metadata querying.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We load the metadata and records from the dataset using `mlcroissant`. The metadata object contains all record set, field, and column definitions, which we will reference by their `@id` attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Hide pandas warnings for clean output

# The Croissant JSON-LD metadata file
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Dataset (Croissant schema)
dataset = mlc.Dataset(croissant_url)
# Metadata is an object with attributes
metadata = dataset.metadata  # Not a dictionary, do not subscript

print(f"Dataset title: {getattr(metadata, 'name', 'No name')}\nDescription: {getattr(metadata, 'description', 'No description')}")

## 2. Data Overview

Let's review the available record sets, fields, and their IDs. Record sets, fields, and columns all have unique `@id` fields which identify them unambiguously within the dataset.

In [ ]:
# List all record sets and their field @ids
from pprint import pprint
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    rs = metadata.record_set
    if isinstance(rs, list):
        record_sets = rs
    elif rs is not None:
        record_sets = [rs]
else:
    record_sets = []
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"Record Set @id: {getattr(rs, '@id', None)}\n  Name: {getattr(rs, 'name', '')}")
    print(f"  Contains fields:")
    if hasattr(rs, 'fields') and rs.fields:
        for f in rs.fields:
            print(f"    - Field @id: {getattr(f, '@id', None)}, Name: {getattr(f, 'name', '')}, Data type: {getattr(f, 'data_type', None)}")
    elif hasattr(rs, 'field') and rs.field:
        fs = rs.field
        if isinstance(fs, list):
            for f in fs:
                print(f"    - Field @id: {getattr(f, '@id', None)}, Name: {getattr(f, 'name', '')}, Data type: {getattr(f, 'data_type', None)}")
        else:
            f = fs
            print(f"    - Field @id: {getattr(f, '@id', None)}, Name: {getattr(f, 'name', '')}, Data type: {getattr(f, 'data_type', None)}")
    else:
        print("    (No fields listed in metadata)")
# If there are no record sets, print that finding
if not record_sets:
    print("No record sets defined in the metadata.")

## 3. Data Extraction

We now load data from a specific record set into a DataFrame for analysis. All selection is performed by referencing the record set and field `@id`s.

Use the printout above to identify the record set(s) of interest.

In [ ]:
# --- Adjust this block for your dataset record set ---

# Let's enumerate available record sets programmatically and pick typical choices for survey data
# If the first record set is the main survey data table, use it

record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Use the mlcroissant API to load all records for the chosen record set @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        print(f"Loaded {len(records)} records from record set {record_set_id}")
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}:")
        print(df.columns.tolist())
        # Show a sample
        display(df.head())
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# For ease of reference below, select the main record set @id (first in list for example purposes)
if record_set_ids:
    main_rs_id = record_set_ids[0]
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field by its `@id`, filter records, normalize values, and perform group-by operations to prepare data for further analysis.

> **Note:** All field references are by their `@id`, not their labels/names.

In [ ]:
import numpy as np

# Choose a numeric field @id for demonstration
# E.g., for a survey, GAD-7 total score or PHQ-9 total score are typical numeric fields
# We'll try common ones, or print columns and let user adjust as needed
df = dataframes.get(main_rs_id, pd.DataFrame())

# Try to auto-select a numeric column typical for this survey
auto_candidates = [col for col in df.columns \
                  if any(key in col.lower() for key in ['phq', 'gad', 'psq', 'score']) and np.issubdtype(df[col].dtype, np.number)]
if auto_candidates:
    numeric_field_id = auto_candidates[0]  # pick the first matching field
else:
    # Fallback: first numeric column
    numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None

print(f"Using numeric field '@id' for analysis: {numeric_field_id}")

if numeric_field_id is not None:
    # Filtering on arbitrary threshold (e.g., score > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Add a normalized column
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std

    print(f"Normalized '{numeric_field_id}' column example:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try group by a categorical field, eg: 'gender', 'level_of_education', etc (by @id, not display name)
    group_candidates = [col for col in df.columns if any(g in col.lower() for g in ['gender', 'education', 'village', 'marital'])]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by field '@id': {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Mean of filtered numeric field by group:")
        display(grouped_df.head())
    else:
        print("No common group field found for groupby analysis.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

We'll visualize data distributions and relationships using seaborn/matplotlib. All field references will still use their `@id` values. Adjust field IDs as needed based on what was printed above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the selected numeric field
if numeric_field_id is not None and not df.empty:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if applicable
    if 'group_field_id' in locals():
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field or data to plot.")

## 6. Conclusion

We have loaded, explored, transformed, and visualized the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. By referencing all entities by their `@id` fields, this notebook ensures reproducible, schema-aware analysis. Examine the output fields above for more in-depth, domain-specific analysis.
